In [1]:
!pip install hnswlib
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679800 sha256=5b475cdd957414296576f6310df4c9437a052f455b4afaa7766836b76e762c53
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [2]:
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # search for 5 nearest neighbors

In [3]:
# Build NN Model
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

# Compute time performance
start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Time:", end - start, "s")

=== Exact NN ===
Indices: [[899 570 621  53 580]]
Distances: [[0.011268   0.01976216 0.03352228 0.04340459 0.04743408]]
Time: 0.03855729103088379 s


In [4]:
p = hnswlib.Index(space='l2', dim=dim)
p.init_index(max_elements=num_elements, ef_construction=100, M=16)
p.add_items(data)
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Time:", end - start, "s")


=== HNSW ===
Indices: [[899 570 621  53 580]]
Distances: [[0.00012697 0.00039054 0.00112374 0.00188396 0.00224999]]
Time: 0.00016307830810546875 s
